# Module 5: Critic-Refiner

Apply **Pattern 3**: add a quality gate. The Writer drafts the memo, the Critic evaluates it against a checklist, and if the bar is not met the memo cycles back for revision: until approved.

![Critic-Refiner: Writer drafts memo → Critic checks 5 criteria → REVISION NEEDED loops back to Writer → APPROVED exits to final memo](./architecture.png)

**When to use this pattern:**
- Quality matters more than speed
- Output can be objectively critiqued against a checklist
- Iteration measurably improves results

**Key Strands primitive:** `GraphBuilder` + cycle edge with `condition=` function + `set_entry_point()`

**Prerequisites:** Modules 1–4. This module reuses tools from Module 2.

## Agents in This Module

| Component | Type | What it does |
|-----------|------|-------------|
| `writer` (graph node) | Agent | Drafts or revises the executive memo based on the brief and any revision feedback |
| `critic` (graph node) | Agent | Evaluates memo quality against 5 criteria; outputs `APPROVED` or `REVISION NEEDED: ...` |

> **Why the Critic must output a fixed signal:** The condition functions (`needs_revision`, `is_approved`) parse the critic's text. If the Critic writes prose instead of the exact signal, the conditions fail silently. The system prompt must force the output format.

In [1]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# ── Model configuration ──────────────────────────────────────────────────
# Option 1, Claude Sonnet 4 (default):
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
# Option 2, Claude Haiku 4.5 (faster):
#   model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")
# Option 3, Amazon Nova Pro (AWS credits):
#   model = BedrockModel(model_id="amazon.nova-pro-v1:0")
# Option 4, Amazon Nova Lite (cheapest):
#   model = BedrockModel(model_id="amazon.nova-lite-v1:0")

✅ Setup complete!


In [ ]:
import sys, os, time

from strands import Agent
from strands.multiagent import GraphBuilder

---

## Part 1: Define the Decision Brief

The brief is passed directly to the `GraphBuilder` cycle. No pre-processing step outside the graph.

In [4]:
DECISION_BRIEF = '''
DECISION BRIEF: NovaCart Premium Tier Launch

Options:
  Option A: Exclusive Premium: invite-only for top 10% of spenders, $19.99/mo
  Option B: Gradual Rollout: 5% A/B test pilot with kill-switch, $14.99/mo
  Option C: Full Launch: open to all users immediately, $12.99/mo + 30-day free trial

Success target: +15% CLV improvement within 6 months
Budget: $2M  |  Decision deadline: 2027-01-31
'''

Decision brief ready.


---

## Part 2: Define the Writer and Critic

The **Writer** drafts or revises the memo. The **Critic** evaluates it against a fixed checklist.

> **Critical design decision:** The Critic's system prompt must force it to start with `APPROVED` or `REVISION NEEDED:`: nothing else. The condition functions parse this signal. If the Critic writes prose, the graph can loop forever.

In [5]:
WRITER_PROMPT = (
    "You are an executive memo writer. Write a COMPLETE leadership memo with exactly these 5 labeled sections:\n## Recommendation (one sentence: which option and why)\n## Options at a Glance (table comparing A, B, C on Complexity/Risk/Verdict)\n## Top 3 Risks (each risk with a specific mitigation action)\n## Success Metrics (at least 2 KPIs with numeric targets)\n## Decision Required (owner, deadline, who must approve)\nIf you receive feedback, revise and include ALL 5 sections in the new version."
)

CRITIC_PROMPT = (
    "You are a quality critic. Check ONLY these 5 criteria:\n1. '## Recommendation' section with a clear option choice (A, B, or C)\n2. '## Options at a Glance' table comparing A, B, C\n3. '## Top 3 Risks' with at least 3 risks each with a specific mitigation\n4. '## Success Metrics' with at least 2 KPIs that have numeric targets\n5. '## Decision Required' with both owner AND deadline\nDO NOT check for research data, background context, or any section not listed.\nRespond with EXACTLY one of:\nAPPROVED\n(if all 5 criteria are met)\nREVISION NEEDED: [which criterion numbers are missing or incomplete]\nYour response must start with APPROVED or REVISION NEEDED."
)

writer = Agent(name="writer", system_prompt=WRITER_PROMPT, callback_handler=None)
critic = Agent(name="critic", system_prompt=CRITIC_PROMPT, callback_handler=None)

---

## Part 3: Build the Graph with Cycle

The condition functions parse the Critic's output to decide the next edge.
`set_entry_point("writer")` is required when a cycle creates ambiguity about which node starts first.

In [6]:
def needs_revision(state):
    '''Returns True if the Critic said REVISION NEEDED.'''
    r = state.results.get("critic")
    return bool(r) and "revision needed" in str(r.result).lower()

def is_approved(state):
    '''Returns True if the Critic said APPROVED.'''
    r = state.results.get("critic")
    return bool(r) and str(r.result).strip().upper().startswith("APPROVED")

builder = GraphBuilder()
builder.add_node(writer, "writer")
builder.add_node(critic, "critic")

builder.set_entry_point("writer")               # required: cycle creates ambiguity

builder.add_edge("writer", "critic")            # writer → critic (always)
builder.add_edge("critic", "writer", condition=needs_revision)  # cycle: revision → writer

# Safety limits, prevent infinite loops
builder.set_max_node_executions(8)              # max 4 writer+critic cycles
builder.set_execution_timeout(180)              # 3 min hard timeout
builder.reset_on_revisit(True)                  # fresh context on each revisit

graph = builder.build()

Graph built. Nodes: ['writer', 'critic']
Edges: writer→critic | critic→writer (if REVISION NEEDED)


---

## Part 4: Run the Critic-Refiner Loop

The graph runs until the Critic says `APPROVED` or the safety limits are reached.

In [7]:
import time

t1 = time.time()
graph_result = graph(DECISION_BRIEF)

total_time = time.time() - t1

# Show each Critic verdict
for node in graph_result.execution_order:
    if node.node_id == "critic":
        verdict = str(node.result)[:120].strip()

Status: Status.COMPLETED | 25.6s
Execution order: ['writer', 'critic']

  Critic verdict: APPROVED


In [8]:
# Print the final approved memo
for node in reversed(graph_result.execution_order):
    if node.node_id == "writer":
        break

────────────────────────────────────────────────────────────
# DECISION BRIEF: NovaCart Premium Tier Launch

---

## Recommendation
Select **Option B (Gradual Rollout)** because the controlled 5% A/B test pilot with a built-in kill-switch minimizes financial and reputational exposure while generating real conversion data needed to optimize pricing and features before committing the full $2M budget.

---

## Options at a Glance

| Criterion | Option A: Exclusive Premium | Option B: Gradual Rollout | Option C: Full Launch |
|---|---|---|---|
| **Price Point** | $19.99/mo | $14.99/mo | $12.99/mo + 30-day trial |
| **Complexity** | 🟡 Medium — invite logic, segmentation required | 🟢 Low-Medium — phased infra, A/B tooling | 🔴 High — full-scale infra, support surge |
| **Risk** | 🔴 High — alienates 90% of user base; perception of exclusion | 🟢 Low — contained blast radius; reversible | 🔴 High — no safety net; trial abuse likely |
| **Verdict** | ⚠️ Reject — high churn risk among non-invited l

---

## Part 5: Pipeline Metrics

In [9]:
writer_runs  = sum(1 for n in graph_result.execution_order if n.node_id == "writer")
critic_runs  = sum(1 for n in graph_result.execution_order if n.node_id == "critic")

Stage                 Runs
----------------------------
Writer (drafts)          1
Critic (reviews)         1

Graph result status: Status.COMPLETED

Key insight: the graph ran until quality was confirmed — not until a fixed number of steps.


---

## Key Takeaways

| Concept | What you saw |
|---------|-------------|
| `GraphBuilder` + cycle | A feedback edge with a condition function creates the revision loop |
| `set_entry_point()` | Required when a cycle makes the start node ambiguous |
| `set_max_node_executions()` | Safety limit: prevents infinite loops |
| Fixed signal output | Critic must output `APPROVED` or `REVISION NEEDED:`: condition functions parse this |
| Quality gate | The system runs until the bar is met, not until a fixed number of steps |

---

## What's Next

In **Module 6: Dynamic Swarm**, agents hand off autonomously: no fixed path, no orchestrator. The route emerges at runtime based on what each agent decides to do next.